# Module 3 · .09 — Model Serialization, Registry & Artifact Management

**Prerequisites:** Deep Learning Foundations, Model Training (NB15)  
**Difficulty:** Intermediate-Advanced  
**Estimated Time:** 60-75 minutes

**Architectural Layer:** Model Management & Deployment Prep  
**Toolchain:** PyTorch · SafeTensors · ONNX · MLflow Model Registry  
**Objective:** Bridge the gap between "I trained a model" and "It's ready for production deployment" by mastering secure serialization, cross-platform export, and lifecycle versioning.

---

## 📑 Table of Contents
* [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · The Serialization Problem: Why `.pth` is Dangerous](#1-the-serialization-problem-why-pth-is-dangerous)
* [2 · SafeTensors: Secure & Fast Weight Storage](#2-safetensors-secure-fast-weight-storage)
* [3 · ONNX Export: Cross-Platform Interoperability](#3-onnx-export-cross-platform-interoperability)
* [4 · The Model Registry (MLflow)](#4-the-model-registry-mlflow)
* [5 · The Model Card Pattern](#5-the-model-card-pattern)
* [✅ Knowledge Check](#knowledge-check)
* [🔨 Practical Practice](#practical-practice)
* [🎯 Summary & Next Steps](#summary--next-steps)

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Juniors train a model, hit `torch.save(model, "model.pth")`, and throw it over the wall to engineers. Seniors know that `.pth` (Pickle) can execute arbitrary malicious code when loaded. They know that a PyTorch dependency is too heavy for edge devices, and they know that without a proper Model Registry, you'll eventually lose track of which weights belong to which production release.

**Mental Model:** 
- **SafeTensors = Secure Hard Drive for Weights:** Fast, safe, zero-copy.
- **ONNX = Universal Translator:** Compiles your model into a math graph independent of PyTorch/TensorFlow.
- **Model Registry = GitHub for Models:** Tracks versions, stages (Staging → Production), and artifacts.

**Common Junior Pitfall:** Assuming deployment means running a Flask server that imports `torch`. In reality, inference infrastructure is highly optimized (TensorRT/ONNXRuntime) and rarely runs native PyTorch.


In [ ]:
# Install required libraries
!pip install -q torch safetensors onnx onnxruntime mlflow pydantic tabulate
print("✅ Libraries installed.")
    


---
## 1 · The Serialization Problem: Why `.pth` is Dangerous

By default, PyTorch uses Python's native `pickle` module to save models. `pickle` was never designed to be secure. When you unpickle a file, it can execute arbitrary Python code. If you download a `.pth` file from the internet, it could contain malware.

Furthermore, `pickle` stores class mappings. If the class definition in the deployment codebase differs slightly from the training codebase, the model will fail to load.


In [ ]:
import torch
import torch.nn as nn
import os

# A simple model architecture
class SimpleClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Linear(10, 2)
        
    def forward(self, x):
        return self.fc(x)

model = SimpleClassifier()

# ❌ BAD PRACTICE: Saving the entire object (highly coupled to the exact Python class)
torch.save(model, "bad_practice_entire_model.pth")

# ⚠️ BETTER BUT STILL INSECURE: Saving just the state_dict using pickle
torch.save(model.state_dict(), "state_dict.pth")

print(f"File sizes:")
print(f"  Entire Model: {os.path.getsize('bad_practice_entire_model.pth')} bytes")
print(f"  State Dict:   {os.path.getsize('state_dict.pth')} bytes")
print("Note: Both still use the insecure Python pickle format underneath.")
    


---
## 2 · SafeTensors: Secure & Fast Weight Storage

Developed by HuggingFace, **SafeTensors** replaces `pickle` entirely.
1. **Security:** It is mathematically guaranteed not to execute code.
2. **Speed:** It supports zero-copy loading (it memory-maps the file directly into RAM/VRAM).
3. **Lazy Loading:** You can load just *one* specific weight matrix without loading the whole 100GB model.


In [ ]:
from safetensors.torch import save_file, load_file
import time

# Create a much larger dummy weight dictionary to see the speed difference
large_weights = {"matrix": torch.randn(5000, 5000)} # ~100MB

# --- Benchmarking Pickle ---
start = time.time()
torch.save(large_weights, "large.pth")
pickle_save_time = time.time() - start

start = time.time()
loaded_pickle = torch.load("large.pth")
pickle_load_time = time.time() - start

# --- Benchmarking SafeTensors ---
start = time.time()
save_file(large_weights, "large.safetensors")
st_save_time = time.time() - start

start = time.time()
loaded_st = load_file("large.safetensors")
st_load_time = time.time() - start

print(f"💾 Saving - Pickle: {pickle_save_time:.4f}s | SafeTensors: {st_save_time:.4f}s")
print(f"🚀 Loading - Pickle: {pickle_load_time:.4f}s | SafeTensors: {st_load_time:.4f}s")
print("\n✅ SafeTensors is inherently safer and often faster via zero-copy mmap.")
    


---
## 3 · ONNX Export: Cross-Platform Interoperability

**ONNX (Open Neural Network Exchange)** is an open standard format for machine learning models.

**Why use ONNX?**
- You trained in PyTorch, but the edge device only runs C++ or Rust.
- You want to use hardware-specific accelerators (NVIDIA TensorRT, Intel OpenVINO, Apple CoreML).
- You want to minimize Docker image size by dropping the massive PyTorch dependency in production and using the lightweight `onnxruntime` instead.

During export, PyTorch runs a dummy input through your model and "traces" the mathematical operations to build a static graph. Let's see it in action.


In [ ]:
import onnx
import onnxruntime as ort
import numpy as np

# We must provide a "dummy input" so PyTorch can trace the exact mathematical operations
dummy_input = torch.randn(1, 10)

# Export to ONNX
onnx_path = "model.onnx"
torch.onnx.export(
    model,                      # model being run
    dummy_input,                # model input (or a tuple for multiple inputs)
    onnx_path,                  # where to save the model
    export_params=True,         # store the trained parameter weights inside the model file
    opset_version=14,           # the ONNX version to export the model to
    do_constant_folding=True,   # whether to execute constant folding for optimization
    input_names=['input'],      # the model's input names
    output_names=['output'],    # the model's output names
    dynamic_axes={              # variable length axes (e.g. batch size)
        'input': {0: 'batch_size'},    
        'output': {0: 'batch_size'}
    }
)

print(f"✅ Model successfully exported to ONNX -> {onnx_path}")

# Verify the ONNX model is well-formed
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print("✅ ONNX model graph verified successfully.")

# Run inference using ONNX Runtime (No PyTorch needed here!)
ort_session = ort.InferenceSession(onnx_path)

def to_numpy(tensor):
    return tensor.detach().cpu().numpy() if tensor.requires_grad else tensor.cpu().numpy()

# Test the inference matching
ort_inputs = {ort_session.get_inputs()[0].name: to_numpy(dummy_input)}
ort_outs = ort_session.run(None, ort_inputs)

# Compare PyTorch vs ONNX Runtime
pt_out = model(dummy_input)
np.testing.assert_allclose(to_numpy(pt_out), ort_outs[0], rtol=1e-03, atol=1e-05)

print("🎯 SUCCESS: ONNX inference outputs match native PyTorch outputs exactly!")
print("   In production, we can now use just the `onnxruntime` library, saving massive overhead.")
    


---
## 4 · The Model Registry (MLflow)

Saving files to disk is fine for personal projects. For teams, you need a **Model Registry**.
A registry is a centralized hub that tracks:
1. **Lineage:** Which training code/hyperparameters produced this artifact?
2. **Versioning:** Is this `v1`, `v2`, or `v3`?
3. **Lifecycle States:** `None` → `Staging` → `Production` → `Archived`.


In [ ]:
import mlflow
import mlflow.pytorch
import tempfile
import urllib.request
from mlflow.tracking import MlflowClient

# 1. Start a local MLflow server logic in memory (sqlite)
mlflow.set_tracking_uri("sqlite:///mlruns.db")
mlflow.set_experiment("fraud_detection_experiment")

# 2. Simulate a full training and logging run
with mlflow.start_run() as run:
    # Log hyperparameters
    mlflow.log_params({"learning_rate": 0.01, "epochs": 10})
    
    # Log performance metrics
    mlflow.log_metrics({"accuracy": 0.95, "f1_score": 0.92})
    
    # 3. Log the PyTorch model AND register it simultaneously!
    # By providing registered_model_name, MLflow auto-creates a versioned entry in the Registry.
    mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        registered_model_name="FraudDetectionModel"
    )

print(f"✅ Model registered! Run ID: {run.info.run_id}")

# 4. Interact with the Model Registry via MLflow Client
client = MlflowClient()

# Fetch latest version of the model
latest_version_info = client.get_latest_versions("FraudDetectionModel", stages=["None"])[0]
model_version = latest_version_info.version

print(f"🔍 Fetching model metadata from registry...")
print(f"   Name: {latest_version_info.name}")
print(f"   Version: {model_version}")
print(f"   Current Stage: {latest_version_info.current_stage}")

# 5. Transition the model from 'None' to 'Staging'
client.transition_model_version_stage(
    name="FraudDetectionModel",
    version=model_version,
    stage="Staging",
    archive_existing_versions=False
)

print(f"🚀 Promoting Model v{model_version} to Staging for integration testing!")
    


---
## 5 · The Model Card Pattern

A **Model Card** (popularized by HuggingFace and Google) is the "nutrition label" for your model. It travels with your serialized weights and explicitly defines:
1. **Intended Use:** What is this model *for*? What is it *not for*?
2. **Inputs/Outputs:** Strict schema definitions.
3. **Bias & Limitations:** Known failure modes and demographic inequalities.
4. **Metrics:** Real-world performance on hold-out test sets.

In modern CI/CD, deployment pipelines often *parse* the Model Card as a JSON/YAML file and run gate checks before allowing deployment.


In [ ]:
import json
from pydantic import BaseModel, Field

# Using Pydantic to strictly type our Model Card
class SchemaDefinition(BaseModel):
    shape: list[int] = Field(..., description="Tensor shape including batch dim")
    dtype: str = Field(..., description="Data type (e.g., float32, int64)")

class ModelCard(BaseModel):
    model_name: str
    version: str
    author: str
    intended_use: str
    out_of_scope_use: str
    limitations: str
    input_schema: SchemaDefinition
    output_schema: SchemaDefinition
    metrics: dict[str, float]

card = ModelCard(
    model_name="FraudDetection_v1",
    version="1.0.0",
    author="Risk ML Team",
    intended_use="Detecting anomalous transaction patterns in real-time user checkout.",
    out_of_scope_use="Automated banning of user accounts without human review.",
    limitations="Model under-indexes on transactions originating from rural locales.",
    input_schema=SchemaDefinition(shape=[-1, 10], dtype="float32"),
    output_schema=SchemaDefinition(shape=[-1, 2], dtype="float32"),
    metrics={"auc_roc": 0.94, "false_positive_rate": 0.02}
)

# Serialize the card to ship alongside the ONNX file
with open("model_card.json", "w") as f:
    f.write(card.model_dump_json(indent=2))

print("📄 Generated Model Card:")
print(card.model_dump_json(indent=2))
print("\n✅ Artifact Bundle is complete: model.onnx + model_card.json")
    


---
## ✅ Knowledge Check

1. **Why shouldn't you share PyTorch models using standard `.pth` (Pickle) files over the public internet?**
<details>
<summary>Answer</summary>
Pickle is essentially a script of Python bytecodes that reconstruct objects. When loaded, a maliciously crafted pickle file can execute arbitrary system commands, leading to total machine compromise. Use SafeTensors instead.
</details>

2. **What does ONNX `dynamic_axes` do, and why is it essential for most deployments?**
<details>
<summary>Answer</summary>
ONNX traces exact operations. Without `dynamic_axes`, your model will only accept an exact batch size (e.g., `batch_size=1`). Setting `dynamic_axes` allows the model to process variable batch sizes in production, which is essential for dynamic HTTP pooling.
</details>

3. **What is the difference between an Artifact repo (like S3) and a Model Registry (like MLflow)?**
<details>
<summary>Answer</summary>
S3 is raw storage. A Model Registry adds an abstraction layer linking artifacts to state machines (Staging, Production), version histories, metrics, and lineage tracking back to the exact code run that produced the weights.
</details>

---

## 🔨 Practical Practice

**Exercise 1: SafeTensors conversion script.** 
Imagine you are given a legacy `weights.pth` file. Write a standard Python utility function that uses PyTorch to load the `.pth` file and immediately saves it via `safetensors.torch.save_file()`.

**Exercise 2: Add dynamic sequence length.** 
Look at the ONNX export code in Cell 3. If we were exporting an NLP Transformer model that has an input shape of `(batch_size, sequence_length)`, how would you update `dynamic_axes` to ensure both batch size AND sequence length can vary in production?

**Exercise 3: MLflow Fetching.**
Using the `client` object initialized in Cell 4, write a command to fetch the exact S3/local URI of the model currently marked as `"Staging"`. 
*(Hint: `client.get_latest_versions("FraudDetectionModel", stages=["Staging"])`)*

---

## 🎯 Summary & Next Steps

### We accomplished:
1. Understanding the grave security risks of `pickle`-based `.pth` files.
2. Adopting HuggingFace's **SafeTensors** for zero-copy, secure weight storage.
3. Decoupling our model from Python using **ONNX**, ensuring portable, C++ optimized inference capabilities.
4. Elevating our storage logic from raw files into a lifecycle-tracked **MLflow Model Registry**.
5. Emmiting professional-grade metadata using the **Model Card** pattern.

### Next Path →
With your model safely bundled, serialized, and versioned, you are now ready to serve it.
**Proceed to → `26_model_context_protocol.ipynb`** (or main serving notebooks) for high-performance deployment strategies.
